# Task 7 — Definition of done (10 checks)

All ten checks, read from the model, not assumed. The character question at
the end: the `-1` counts are read off the fact table, not from joins — and
deleting the 14 city rows would have "gained" a green dashboard and lost
real freight.

Run all cells top to bottom from the `Task6/` folder.


In [1]:
import importlib.util
from pathlib import Path
import pandas as pd

NB_DIR = Path.cwd()
DATA = NB_DIR / "data"
print("notebook dir:", NB_DIR)
print("data dir exists:", DATA.exists())

# The pipeline script's helpers, loaded from the file sitting next to this
# notebook — the same strict rules as W6_su92-bsdsm-f23-018.py: plain-decimal
# numbers only, DD/MM/YYYY-or-ISO dates, positional extract (headers differ
# across the three files, so columns are read by position, not by name).
_spec = importlib.util.spec_from_file_location("w6pipe", NB_DIR / "W6_su92-bsdsm-f23-018.py")
w6 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(w6)
CONTRACT, PLAIN, STATUS_MAP = w6.CONTRACT, w6.PLAIN, w6.STATUS_MAP
norm_id, plain_number, parse_date = w6.norm_id, w6.plain_number, w6.parse_date
extract_csv, extract_rail = w6.extract_csv, w6.extract_rail
find_source_files, mode_key, bizkey = w6.find_source_files, w6.mode_key, w6.bizkey
print("pipeline helpers loaded from W6_su92-bsdsm-f23-018.py")

notebook dir: /home/hatch/workspace/data-warehouse/week6/Task6
data dir exists: True
pipeline helpers loaded from W6_su92-bsdsm-f23-018.py


In [2]:
from collections import Counter

marine_path, rail_path, road_path = find_source_files(DATA / "march")
staging, artifacts = [], []
for path, kind in ((marine_path, "csv"), (rail_path, "xlsx"), (road_path, "csv")):
    rows, arts = extract_csv(path, path.name) if kind == "csv" else extract_rail(path, path.name)
    staging.extend(rows)
    artifacts.extend(arts)
    print(f"  {path.name}: {len(rows)} rows")
for a in artifacts:
    print(f"  extract artifact (NOT a row): {a[0]} line {a[1]} — {a[2]}")
stg = pd.DataFrame(staging)[CONTRACT + ["SourceFile", "SourceLine", "SourceRow"]]
print("per-file rows:", dict(Counter(stg["SourceFile"])))
print("staging rows:", len(stg))
assert len(stg) == 1284, "staging must be 1,284 rows"


  marine_march.csv: 261 rows
  rail_march.xlsx: 315 rows
  road_march.csv: 708 rows
  extract artifact (NOT a row): road_march.csv line 710 — blank line
  extract artifact (NOT a row): road_march.csv line 711 — TOTAL footer
per-file rows: {'marine_march.csv': 261, 'rail_march.xlsx': 315, 'road_march.csv': 708}
staging rows: 1284


In [3]:
alias = {r["RawSpelling"].strip(): int(r["CityKey"]) for r in
         pd.read_csv(DATA / "alias_lookup.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_city = pd.read_csv(DATA / "dims" / "DimCity.csv", dtype=str, keep_default_na=False)
city_name = {int(r["CityKey"]): r["CityName"] for r in dim_city.to_dict("records")}
route = {(int(r["OriginCityKey"]), int(r["DestCityKey"])): int(r["RouteKey"]) for r in
         pd.read_csv(DATA / "dims" / "DimRoute.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_status = pd.read_csv(DATA / "dims" / "DimStatus.csv", dtype=str, keep_default_na=False)
status_names = {int(r["StatusKey"]): r["StatusName"] for r in dim_status.to_dict("records")}
print(f"alias spellings: {len(alias)}; cities: {len(city_name)}; routes: {len(route)}; statuses: {status_names}")

alias spellings: 23; cities: 15; routes: 21; statuses: {1: 'Delivered', 2: 'In Transit', 3: 'Delayed', -1: 'Unknown'}


In [4]:
# ---- REJECT: the four classes (disjoint for March; verified below) ----
wtext = stg["WeightKG"].map(lambda v: plain_number(v) is None and (v or "").strip() != "")
baddate = stg["ShipDate"].map(lambda v: parse_date(v) is None)
wneg = stg["WeightKG"].map(lambda v: plain_number(v) is not None and plain_number(v) < 0)
blankid = stg["ShipmentID"].map(lambda v: norm_id(v) == "")
assert not (wtext & baddate).any() and not (wtext & wneg).any() and not (wtext & blankid).any()
assert not (baddate & wneg).any() and not (baddate & blankid).any() and not (wneg & blankid).any()
rejected = stg[wtext | baddate | wneg | blankid].copy()
loadable = stg[~(wtext | baddate | wneg | blankid)].copy()
print(f"rejected: {len(rejected)} (weight-text {int(wtext.sum())}, date-unparseable {int(baddate.sum())}, "
      f"weight-negative {int(wneg.sum())}, id-blank {int(blankid.sum())}); loadable: {len(loadable)}")

# ---- DEDUP: TRIM + UPPER ShipmentID, first occurrence kept ----
loadable["_n"] = loadable["ShipmentID"].map(norm_id)
seen_raw, byte_removed = set(), 0
for _, row in loadable.iterrows():
    k = tuple(row[c] for c in CONTRACT)
    if k in seen_raw:
        byte_removed += 1
    else:
        seen_raw.add(k)
deduped = loadable.drop_duplicates(subset=["_n"], keep="first").copy()
removed = len(loadable) - len(deduped)
print(f"duplicates removed: {removed} ({byte_removed} byte-identical + {removed - byte_removed} trailing-space-only)")
deduped = deduped.drop(columns=["_n"])

# ---- TRANSFORM + ROUTE ----
work = deduped.copy()
is_rail = work["SourceFile"].map(lambda f: "rail" in f.lower())
wnum = work["WeightKG"].map(plain_number)
wkg = wnum.copy()
wkg[is_rail] = wkg[is_rail] * 1000
work["WeightKG_num"] = wkg.map(lambda v: int(round(v)))
work["FreightCostPKR_num"] = work["FreightCostPKR"].map(lambda v: int(round(plain_number(v))))
work["ShipmentID"] = work.apply(
    lambda r: ("RL-" + norm_id(r["ShipmentID"])
               if "rail" in r["SourceFile"].lower() and not norm_id(r["ShipmentID"]).startswith("RL-")
               else norm_id(r["ShipmentID"])), axis=1)
work["ShipDate_parsed"] = work["ShipDate"].map(parse_date)
work["DateKey"] = work["ShipDate_parsed"].map(lambda d: int(d.strftime("%Y%m%d")))
work["StatusKey"] = work["StatusRaw"].map(lambda v: STATUS_MAP.get((v or "").strip().upper(), ("", -1))[1])
work["OriginCityKey"] = work["Origin"].map(lambda v: alias.get((v or "").strip(), -1))
work["DestCityKey"] = work["Destination"].map(lambda v: alias.get((v or "").strip(), -1))
work["RouteKey"] = [route.get((o, d), -1) for o, d in zip(work["OriginCityKey"], work["DestCityKey"])]
work["ModeKey"] = work["SourceFile"].map(mode_key)

# ---- CONFORM CUSTOMERS: merge registry duplicates, keep the losing spelling as an alias ----
cust = pd.read_csv(DATA / "customers_master.csv", dtype=str, keep_default_na=False)
cust["num"] = cust["CustomerID"].map(lambda v: int(v.split("-")[1]))
cust["biz"] = cust["CustomerName"].map(bizkey)
groups = {}
for _, r in cust.iterrows():
    groups.setdefault(r["biz"], []).append(r)
alias_map, dim_rows = {}, []
for biz, members in groups.items():
    members = sorted(members, key=lambda r: r["num"])
    surv = members[0]
    for m in members[1:]:
        alias_map[m["num"]] = surv["num"]
    dim_rows.append({"CustomerKey": int(surv["num"]), "CustomerName": surv["CustomerName"],
                     "City": surv["City"], "Sector": surv["Sector"],
                     "Aliases": "; ".join(m["CustomerName"] for m in members[1:])})
work["CustomerKey"] = work["CustomerID"].map(
    lambda v: alias_map.get(int(v.split("-")[1]), int(v.split("-")[1])))
dim_customer = pd.DataFrame(dim_rows).sort_values("CustomerKey").reset_index(drop=True)
dim_customer = pd.concat([dim_customer, pd.DataFrame(
    [{"CustomerKey": -1, "CustomerName": "Unknown", "City": "Unknown", "Sector": "Unknown", "Aliases": ""}])],
    ignore_index=True)
print(f"registry rows: {len(cust)}; businesses after conform: {len(dim_customer) - 1}; merged pairs: {len(alias_map)}")

# ---- FACT (in memory; the pipeline script writes this to CSV) ----
fact = pd.DataFrame({
    "ShipmentID": work["ShipmentID"],
    "DateKey": work["DateKey"].astype(int),
    "OriginCityKey": work["OriginCityKey"].astype(int),
    "DestCityKey": work["DestCityKey"].astype(int),
    "RouteKey": work["RouteKey"].astype(int),
    "ModeKey": work["ModeKey"].astype(int),
    "StatusKey": work["StatusKey"].astype(int),
    "CustomerKey": work["CustomerKey"].astype(int),
    "WeightKG": work["WeightKG_num"].astype(int),
    "FreightCostPKR": work["FreightCostPKR_num"].astype(int),
    "ShipmentCount": 1})
print(f"fact rows: {len(fact)}; total weight: {int(fact['WeightKG'].sum()):,} kg; "
      f"total cost: PKR {int(fact['FreightCostPKR'].sum()):,}")

rejected: 24 (weight-text 9, date-unparseable 6, weight-negative 5, id-blank 4); loadable: 1260
duplicates removed: 60 (48 byte-identical + 12 trailing-space-only)
registry rows: 46; businesses after conform: 40; merged pairs: 6
fact rows: 1200; total weight: 17,542,050 kg; total cost: PKR 147,037,000


In [5]:
# All ten checks, each read from the staged / routed / fact data.
dim_date = w6.build_dimdate_q1()
checks = pd.DataFrame([
    ("rows extracted into staging", len(stg), 1284),
    ("rows in RejectLog", len(rejected), 24),
    ("duplicates removed", removed, 60),
    ("rows in FactShipment_March", len(fact), 1200),
    ("SUM WeightKG", int(fact["WeightKG"].sum()), 17542050),
    ("SUM FreightCostPKR", int(fact["FreightCostPKR"].sum()), 147037000),
    ("rows with OriginCityKey = -1 (Unknown city)", int((fact["OriginCityKey"] == -1).sum()), 14),
    ("rows with StatusKey = -1 (Unknown status)", int((fact["StatusKey"] == -1).sum()), 7),
    ("DimCustomer members excluding Unknown", len(dim_customer) - 1, 40),
    ("rows in DimDate", len(dim_date), 90),
], columns=["Check", "Got", "Target"])
checks["Result"] = ["PASS" if g == t else "FAIL" for g, t in zip(checks["Got"], checks["Target"])]
print(checks.to_string(index=False))
identity = len(stg) - len(fact) - len(rejected) - removed
print(f"\nidentity 1 = 2 + 3 + 4: {len(stg)} = {len(rejected)} + {removed} + {len(fact)}; staging - loaded - rejected - removed = {identity}")

                                      Check       Got    Target Result
                rows extracted into staging      1284      1284   PASS
                          rows in RejectLog        24        24   PASS
                         duplicates removed        60        60   PASS
                 rows in FactShipment_March      1200      1200   PASS
                               SUM WeightKG  17542050  17542050   PASS
                         SUM FreightCostPKR 147037000 147037000   PASS
rows with OriginCityKey = -1 (Unknown city)        14        14   PASS
  rows with StatusKey = -1 (Unknown status)         7         7   PASS
      DimCustomer members excluding Unknown        40        40   PASS
                            rows in DimDate        90        90   PASS

identity 1 = 2 + 3 + 4: 1284 = 24 + 60 + 1200; staging - loaded - rejected - removed = 0


In [6]:
# The character question: name the unresolved values, and price deleting them.
from collections import Counter
unk_o = loadable[loadable["Origin"].map(lambda v: alias.get((v or "").strip()) is None)]
spell_o = Counter(unk_o["Origin"].map(lambda v: (v or "").strip()))
unk_s = loadable[loadable["StatusRaw"].map(lambda v: STATUS_MAP.get((v or "").strip().upper()) is None)]
spell_s = Counter(unk_s["StatusRaw"].map(lambda v: (v or "").strip()))
print("city -1 spellings (14 rows, 6 spellings):", dict(spell_o))
print("status -1 spellings (7 rows, 3 spellings):", dict(spell_s))
city1_w = int(fact.loc[fact["OriginCityKey"] == -1, "WeightKG"].sum())
print(f"deleting the 14 city rows would have produced a tidy {len(fact) - 14:,}-row fact table and hidden {city1_w:,} kg of freight actually moved")

city -1 spellings (14 rows, 6 spellings): {'Keti Bandar Jetty': 3, 'Kotri Junction': 3, 'Wagah Border': 2, 'Taftan Crossing': 2, 'Torkham Gate': 2, 'Chaman Border': 2}
status -1 spellings (7 rows, 3 spellings): {'RTO': 2, 'Hold': 1, 'held at customs': 4}
deleting the 14 city rows would have produced a tidy 1,186-row fact table and hidden 195,150 kg of freight actually moved


In [7]:
assert (checks["Result"] == "PASS").all(), "all ten checks must read PASS"
assert identity == 0
assert dict(spell_o) == {"Keti Bandar Jetty": 3, "Kotri Junction": 3, "Wagah Border": 2,
                         "Taftan Crossing": 2, "Torkham Gate": 2, "Chaman Border": 2}
assert dict(spell_s) == {"held at customs": 4, "RTO": 2, "Hold": 1}
assert city1_w == 195150
print("SELF-CHECK PASS — 10/10 checks pass; identity 0; -1 spellings named; 195,150 kg priced")

# Analysis
# The -1 counts are read off the fact table (filter OriginCityKey = -1 ->
# 14; StatusKey = -1 -> 7), not assumed from joins. The 14 city rows are the
# 6 unresolved spellings above; the 7 status rows are 'held at customs' 4,
# 'RTO' 2, 'Hold' 1. Deleting the 14 city rows would have "gained" a green
# dashboard and lost 195,150 kg of freight the company actually moved — the
# count would read 1,186 and the quarter would be understated with no trace.
# Routing them is what makes the 1,200-row table honest.

SELF-CHECK PASS — 10/10 checks pass; identity 0; -1 spellings named; 195,150 kg priced
